In [1]:
import pandas as pd
import ast

#When I first loaded the credits csv there were ~1600 blank columns.  Pandas had a hard time determining what were legit cols becasue of the comma placements.
#instead I wanted just three cols that can be searched. 
credits_df = pd.read_csv("credits.csv", usecols=[0,1,2], engine="python")
credits_df.columns = ["cast", "crew", "id"]
credits_df.to_csv("clean_credits.csv", index=False)
keyword_df = pd.read_csv("keywords.csv")
movies_df = pd.read_csv("movies_metadata.csv")

/var/folders/km/h432qbx55ljb5h2m9n7_s65c0000gn/T/ipykernel_65813/3294370453.py:10: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_df = pd.read_csv("movies_metadata.csv")


In [ ]:
credits_df.head()

In [ ]:
credits_df.info()

In [2]:

#there are three rows that are causing problems.  There was the wrong data type in several rows that were gumming up the works. 
#I identified these rows by the junk data that was in the first col and then dropped these rows.
junk_data = movies_df[~movies_df['adult'].isin(["TRUE", "FALSE"])].index

movies_df.drop(index=junk_data, inplace=True)

In [ ]:
#Perhaps there was a better way to clean up the data from the csv that I downloaded.
#At some point I was ready to throw my laptop out of the windwow. 
#Instead I turned to Google Gemini to assist with getting a clean copy of the csv so that I can work with it in my project.
#Even this took about 20 iterations. :(
#I went through a few different approaches then decided I wanted to break everything out into separate dataframes and connect to the movie by id

def expand_movie_data(df, column_name):
    # 1. Create a copy with only needed columns
    new_df = df[['id', column_name]].copy()
    
    # 2. Convert strings to lists of dicts safely
    def safe_parse(x):
        if isinstance(x, str):
            try:
                # This handles the SyntaxError by returning an empty list if parsing fails
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return []
        return x

    new_df[column_name] = new_df[column_name].apply(safe_parse)
    
    # 3. Explode the lists into separate rows
    new_df = new_df.explode(column_name)
    
    # 4. Extract only the 'name' from each dictionary
    new_df[column_name] = new_df[column_name].apply(
        lambda x: x.get('name') if isinstance(x, dict) else None
    )
    
    # 5. Drop any rows that couldn't be parsed
    return new_df.dropna(subset=[column_name])

# Create your new dataframes
# Use movies_df for keywords if keyword_df wasn't pre-defined
keyword_df = expand_movie_data(keyword_df, 'keywords')
genres_df = expand_movie_data(movies_df, 'genres')
production_df = expand_movie_data(movies_df, 'production_companies')
crew_df = expand_movie_data(credits_df, 'crew')
cast_df = expand_movie_data(credits_df, 'cast')


In [5]:
keyword_df

,id,keywords
0,862,jealousy
0,862,toy
0,862,boy
0,862,friendship
0,862,friends
...,...,...
46411,289923,mockumentary
46414,439050,tragic love
46415,111109,artist
46415,111109,play


In [6]:
genres_df

,id,genres
0,862,Animation
0,862,Comedy
0,862,Family
1,8844,Adventure
1,8844,Fantasy
...,...,...
45464,302349,Comedy
45464,302349,Fantasy
45464,302349,Science Fiction
45465,47871,Action


In [7]:
production_df

,id,production_companies
0,862,Pixar Animation Studios
1,8844,TriStar Pictures
1,8844,Teitler Film
1,8844,Interscope Communications
2,15602,Warner Bros.
...,...,...
45462,111109,Sine Olivia
45463,67758,American World Pictures
45464,302349,27 Films Production
45464,302349,Potemkino


In [ ]:
cast_df.to_csv('clean_cast.csv', index=False)

crew_df.to_csv('clean_crew.csv', index=False)

movies_df.to_csv('clean_movies.csv', index=False)

keyword_df.to_csv('clean_keyword.csv', index=False)

production_df.to_csv('clean_production.csv', index=False)

genres_df.to_csv('clean_genres.csv', index=False)


In [10]:
movies_df

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,FALSE,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,10/30/95,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,FALSE,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,12/15/95,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,FALSE,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,12/22/95,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,FALSE,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,12/22/95,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,FALSE,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,2/10/95,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45461,FALSE,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,NaN,0.0,90.0,"[{'iso_639_1': 'fa', 'name': 'فارسی'}]",Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0
45462,FALSE,NaN,0,"[{'id': 18, 'name': 'Drama'}]",NaN,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,11/17/11,0.0,360.0,"[{'iso_639_1': 'tl', 'name': ''}]",Released,NaN,Century of Birthing,False,9.0,3.0
45463,FALSE,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",NaN,67758,tt0303758,en,Betrayal,"When one of her hits goes wrong, a professiona...",...,8/1/03,0.0,90.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,A deadly game of wits.,Betrayal,False,3.8,6.0
45464,FALSE,"{'id': 312977, 'name': 'Iron Sky Collection', ...",18000000,"[{'id': 28, 'name': 'Action'}, {'id': 35, 'nam...",http://www.ironsky.net/,302349,tt3038708,en,Iron Sky: The Coming Race,"Twenty years after the events of Iron Sky, the...",...,3/1/18,0.0,0.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Post Production,NaN,Iron Sky: The Coming Race,False,0.0,0.0


In [8]:
cast_df

,id,cast
0,862,Tom Hanks
0,862,Tim Allen
0,862,Don Rickles
0,862,Jim Varney
0,862,Wallace Shawn
...,...,...
45502,227506,Iwan Mosschuchin
45502,227506,Nathalie Lissenko
45502,227506,Pavel Pavlov
45502,227506,Aleksandr Chabrov


In [9]:
crew_df

,id,crew
0,862,John Lasseter
0,862,Joss Whedon
0,862,Andrew Stanton
0,862,Joel Cohen
0,862,Alec Sokolow
...,...,...
45501,67758,Richard McHugh
45501,67758,João Fernandes
45502,227506,Yakov Protazanov
45502,227506,Joseph N. Ermolieff
